In [0]:
%sql

---------------------create owener list-----------------
select distinct cast(CMS.Document_Id as string) as Doc_ID, (case when S1.document_id is not null then true else false end) as scheduled, CL.cluster, CMS.created as Owner, UP.DisplayName, UP.Title, UP.BU, UP.BU_Old, UP.Country, CMS.updated as last_update_by
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(trim(CL.Document_ID))=upper(trim(CMS.Document_Id ))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
on UP.UserName = CMS.created
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as S1
on upper(trim(CMS.Document_Id))=upper(trim(S1.document_id))
where CMS.Document_Id is not null 
  and cms.Document_name not in ('Document Not Found on Server')
  and UP.BU =''

select distinct cast(cluster as string), scheduled, count(distinct Doc_ID) from _sqldf
group by cluster, scheduled
union all 
select "Total" as cluster,  count(distinct Doc_ID) from _sqldf
group by scheduled
order by cluster asc, scheduled asc


select distinct cast(CL.Document_ID as string) as Doc_ID, CL.cluster, CL.created as Owner, UP.DisplayName, UP.Title, UP.BU, UP.BU_Old, UP.Country, CMS.updated as last_update_by
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(trim(CL.Document_ID))=upper(trim(CMS.Document_Id ))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
on UP.UserName = CMS.created
where CMS.created is null

select distinct cms.Document_Id, cms.Document_name,cms.WEBI_Found_on_Server,cms.Full_path,cms.ingestion_ts from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata  as cms
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as sf
on upper(Trim(cms.Document_Id))=upper(trim(sf.Report_ID))
where sf.Report_ID is null
and cms.Document_name not in ('Document Not Found on Server')

select 
-- select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted

In [0]:
%sql
select cluster, BU, scheduled, count(distinct Doc_ID) as document_cnt, count(distinct UserName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
GROUP BY cluster, BU, scheduled
having count(distinct UserName) > 0
;

select count(distinct Doc_ID) from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis


In [0]:
import matplotlib.pyplot as plt
import pandas as pd

pdf = spark.sql("""
select cluster, BU, scheduled, count(distinct Doc_ID) as document_cnt, count(distinct UserName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
GROUP BY cluster, BU, scheduled
having count(distinct UserName) > 0
""").toPandas()

# Pivot: cluster on x-axis, BU as legend
pivot_df = pdf.groupby(['cluster', 'BU'])['document_cnt'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
pivot_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_xlabel('Cluster')
ax.set_ylabel('Document Count')
ax.set_title('Document Count by Cluster and BU')
ax.legend(title='BU', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Pie chart: BU distribution for Cluster 6
cluster6 = pdf[pdf['cluster'] == 6.0].groupby('BU')['document_cnt'].sum()
cluster6 = cluster6[cluster6 > 0].sort_values(ascending=False)

fig2, ax2 = plt.subplots(figsize=(10, 8))
ax2.pie(cluster6, labels=cluster6.index, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
ax2.set_title('Cluster 6 - BU Distribution (Document Count)')
plt.tight_layout()
plt.show()

In [0]:
# --- Configuration ---
BU_Group_1="COMMERCIALS"
bu_list_BG1 = ['COMCE-GERMANY,AUSTRIA, SWITZERL,CEE', 'COMAS-ASIA', 'GLOBA-GLOBAL','COMNA-NORTH AMERICA','COMNN-NETHERLANDS','COMON-OCEANIA','COMSE-BELGIUM, LUXEMBOURG','COMSPB-SPAIN, PORTUGAL, BRAZIL','COMUI-UK & IRELAND','FRANC-FRANCE','ITALY-ITALY','CREDIT INSURANCE' ]

# --- Configuration ---
BU_Group_2="RISK"
bu_list_BG2 = ['RISK1-RS1-GERMANY, CENTRAL & EAST EUROPE','RISK2-RS2-BELGIUM, LUX, FRANCE & ITA','RISK3-RS3-NLD, APAC & AIM','RISK4-RS4-NORTH EUROPE, CIS & ACS','RISK4-RS4-NORTH EUROPE, CIS & SP','RISK5-RS5-NAFTA','RISK1-RS1-DEU, AUT and CHE','RISK1-RS1-Europe, Russia & CIS','RISK2-RS2-NLD, Belux, FRA, Africa & ISR','RISK3-RS3-Asia and Oceania','RISK7-RS7-Spain, Portugal, Andorra']

# --- Configuration ---
BU_Group_3="FINANCE"
bu_list_BG3 = ['FINPM-FINANCE PROGRAM MANAGEMENT','GFO-GROUP FINANCE OPERATIONS','GFC-GROUP FINANCE AND CONTROL','COFIN-CORPORATE FINANCE & TAX','Group Finance','Finance','Finance & Control']

# --- Configuration ---
BU_Group_4="Unidentified"
bu_list_BG4 = ['Unidentified', 'Left Organization']  # Also includes NULL BUs (handled via COALESCE in SQL CASE)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query: Schedule Report Owners by BU (reports per category) ---
pdf_sched_owners = spark.sql("""
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email multiple users'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE sd2.Active_Schedule = TRUE
)
SELECT
  UP.BU,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
  ON sd3.document_id = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE sd2.document_id IS NOT NULL
  AND sd3.Active_Schedule = TRUE
GROUP BY
  UP.BU,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END
ORDER BY Category ASC, Reports_cnt DESC
""").toPandas()

# --- Separate query: Distinct users per BU (no category grouping = no double counting) ---
pdf_bu_users = spark.sql("""
  SELECT
    UP.BU,
    COUNT(DISTINCT CMS.created) AS User_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
  GROUP BY UP.BU
""").toPandas()

bu_user_totals = pdf_bu_users.set_index('BU')['User_cnt'].to_dict()
print(f"Total Users: {sum(bu_user_totals.values())}")
# --- Pivot for stacked bar chart ---
pivot = pdf_sched_owners.pivot_table(index='BU', columns='Category', values='Reports_cnt', aggfunc='sum', fill_value=0)

# Sort by total reports descending
pivot['_total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('_total', ascending=False)
pivot = pivot.drop(columns='_total')

owners = pivot.index
categories = pivot.columns
colors = {
    'Email self': '#5DADE2',
    'Email multiple users': '#1B4F72',
    'Shared Drive': '#F4D03F',
    'SAP BO': '#E67E22',
    'Paused': '#C0C0C0'
}

fig, ax = plt.subplots(figsize=(14, 7))
width = 0.4
bottom = pd.Series([0.0]*len(owners), index=owners)
for cat in categories:
    ax.bar(owners, pivot[cat], width=width, bottom=bottom, color=colors.get(cat, '#888888'), label=cat)
    bottom += pivot[cat]

# Add total users annotation above each bar
for i, bu in enumerate(owners):
    total_height = bottom[bu]
    users = bu_user_totals.get(bu, 0)
    ax.text(i, total_height + 1, f'{int(users)} users', ha='center', va='bottom', fontsize=7, color='#555555')

ax.set_ylabel('Scheduled Reports Count')
ax.set_title('All BUs - Scheduled Reports by Owners BU')
ax.legend(title='Schedule Category', fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Display table ---
display(pdf_sched_owners)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- BU Group mappings from cell 4 ---
bu_group_map = {}
for bu in bu_list_BG1:
    bu_group_map[bu.upper()] = BU_Group_1
for bu in bu_list_BG2:
    bu_group_map[bu.upper()] = BU_Group_2
for bu in bu_list_BG3:
    bu_group_map[bu.upper()] = BU_Group_3
for bu in bu_list_BG4:
    bu_group_map[bu.upper()] = BU_Group_4

# Build SQL CASE expression for BU grouping (applied BEFORE aggregation to avoid double counting)
# NULL BUs also map to Unidentified via COALESCE
case_lines = []
for bu_val, group_name in bu_group_map.items():
    safe_val = bu_val.replace("'", "''")
    case_lines.append(f"WHEN UPPER(UP.BU) = '{safe_val}' THEN '{group_name}'")
bu_group_case = "CASE\n" + "\n".join(case_lines) + f"\nELSE COALESCE(UP.BU, '{BU_Group_4}')\nEND"

# --- Query with BU grouping at SQL level: COUNT(DISTINCT) is correct per group ---
pdf_grouped = spark.sql(f"""
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email multiple users'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Drive'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd2
  ON CAST(sd1.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE sd2.Active_Schedule = TRUE
)
SELECT
  {bu_group_case} AS BU_Group,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
  ON sd3.document_id = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE sd2.document_id IS NOT NULL
  AND sd3.Active_Schedule = TRUE
GROUP BY
  {bu_group_case},
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Drive'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END
ORDER BY BU_Group ASC, Reports_cnt DESC
""").toPandas()

# --- Users per BU Group (also deduplicated at SQL level) ---
pdf_bu_group_users = spark.sql(f"""
  SELECT
    {bu_group_case} AS BU_Group,
    COUNT(DISTINCT CMS.created) AS User_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
  GROUP BY {bu_group_case}
""").toPandas()

bu_group_user_totals = pdf_bu_group_users.set_index('BU_Group')['User_cnt'].to_dict()
print(f"Total Users: {sum(bu_group_user_totals.values())}")

# --- Pivot for stacked bar chart ---
pivot = pdf_grouped.pivot_table(index='BU_Group', columns='Category', values='Reports_cnt', aggfunc='sum', fill_value=0)

# --- Custom sort: 3 main groups first, then others by total desc, Unidentified last ---
pivot['_total'] = pivot.sum(axis=1)

# Define priority groups
priority_front = [BU_Group_1, BU_Group_2, BU_Group_3]  # COMMERCIALS, RISK, FINANCE
priority_end = [BU_Group_4]  # Unidentified (includes NULL, Unidentified, Left Organization)

# Split into front, middle, end
front = [g for g in priority_front if g in pivot.index]
middle = [g for g in pivot.index if g not in priority_front and g not in priority_end]
end = [g for g in priority_end if g in pivot.index]

# Sort front by total desc, middle by total desc
front_sorted = pivot.loc[front].sort_values('_total', ascending=False).index.tolist()
middle_sorted = pivot.loc[middle].sort_values('_total', ascending=False).index.tolist()

# Final order: groups first, others in middle, Unidentified last
final_order = front_sorted + middle_sorted + end
pivot = pivot.loc[final_order]
pivot = pivot.drop(columns='_total')

owners = pivot.index
categories = pivot.columns
colors = {
    'Email self': '#5DADE2',
    'Email multiple users': '#1B4F72',
    'Shared Drive': '#F4D03F',
    'SAP BO': '#E67E22',
    'Paused': '#C0C0C0'
}

fig, ax = plt.subplots(figsize=(14, 7))
width = 0.4
bottom = pd.Series([0.0]*len(owners), index=owners)
for cat in categories:
    ax.bar(owners, pivot[cat], width=width, bottom=bottom, color=colors.get(cat, '#888888'), label=cat)
    bottom += pivot[cat]

# Annotate: owner count above each bar, clearly labeled
for i, bu in enumerate(owners):
    total_height = bottom[bu]
    users = int(bu_group_user_totals.get(bu, 0))
    ax.text(i, total_height + 2, f'{users} user', ha='center', va='bottom',
            fontsize=8, fontstyle='italic', color='black')

ax.set_ylabel('Scheduled Reports Count')
ax.set_title('Scheduled Reports by BU Groups (COMMERCIALS / RISK / FINANCE) and Individual BU')
ax.legend(title='Schedule Category', fontsize=10, ncol=5, loc='upper center',
          bbox_to_anchor=(0.5, 0.98), frameon=True, framealpha=0.9,
          columnspacing=1.0, handletextpad=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Display table ---
display(pdf_grouped)

In [0]:
# --- Summary: Total distinct BUs and Total distinct Reports in this analysis ---
total_bus = spark.sql("""
  SELECT COUNT(DISTINCT UP.BU) AS total_bus
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
""").collect()[0]['total_bus']

total_reports = spark.sql("""
  SELECT COUNT(DISTINCT CMS.Document_Id) AS total_reports
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS sd3
    ON sd3.document_id = sd2.document_id
  WHERE sd2.document_id IS NOT NULL
    AND sd3.Active_Schedule = TRUE
""").collect()[0]['total_reports']

# Reports in Unidentified group (includes NULL BU, 'Unidentified', 'Left Organization')
unidentified_reports = int(pdf_grouped[pdf_grouped['BU_Group'] == BU_Group_4]['Reports_cnt'].sum())

print(f"Total distinct BUs in analysis: {total_bus}")
print(f"Total distinct Scheduled Reports: {total_reports}")
print(f"Reports in '{BU_Group_4}' group (NULL + Unidentified + Left Organization): {unidentified_reports}")